# Figure 4 — Exposure–response relationships for continuous weather exposures

- **FIG_ID**: `Figure 4`（编号真源：`data/figure_numbering.csv`，§4.0；output stem `Figure_4`）
- **图位**: v6 正文行 238（§4.3）；对应 `[FIGURE 4 PLACEHOLDER]`
- **数据依赖**（A 层冻结，SHA-256 见 `data/figure_data_manifest.json`）:
  - `data/fig04_curve_grids.csv`（曲线 + delta 法点态 95% 带）
  - `data/fig04_threshold_points.csv`（(a) 阈值叠加点，A 层登记值）
  - `data/fig04_exposure_hist.csv`（底部暴露分布条带）
  - `data/fig04_annotations.csv`（参考值/稀疏分位/观测数/联合 Wald p/断点检索证据）
  - `data/fig04_plot_windows.csv`（绘制窗口与纵轴范围）
- **上游变更提示**: 若 B 层完整案例、主面板或 A 层登记表更新，先运行
  `python scripts/extract_fig04_data.py` 重新拟合并冻结，再执行本册（Cell 3 会拦截哈希不匹配）
- **输出**: `out/draft/Figure_4.{png,pdf,eps}`（`FIG_DRAFT=1`，默认）/ `out/submission/Figure_4.*`
  （`FIG_DRAFT=0`）+ `out/captions/Figure_4_caption.txt` + `qc/figure_4_qc.json`
- **图面约束**（v6 行 238）: (a) 降水剂量–反应（横轴 log 变换日降水），叠加 ≥ 25 mm 与 ≥ 50 mm
  阈值点，注明"未发现稳定刚性断点"；(b) 日最小能见度剂量–反应，参考点 = 晴天中位数；
  两 panel 均为 length-offset 等价规格、delta 法 95% 点态带、数据稀疏区加阴影并标为描述性
- **规格**: 冻结 SPEC_P / M3（section 固定效应 + month + dow + holiday + 时间趋势 + log flow），
  NB2（alpha 估计），section 聚类 CR1；连续暴露用 `bs(·, df=4)` 替换线性项
- **设计系统（计划书 §1.6，v2.3.1）**: 2×2 结构（左列降水 / 右列最小能见度；上排 RR / 下排暴露分布）；
  主色 NJU purple `#6E0F6D`、95% 带 `#E8D7E8`、阈值估计 `#D89000`、参考线 `#4A4A4A`、
  稀疏区 `#F2EEF2`、分布条带中性灰 `#D9D9D9`；字号取自 `fig_common.DS`（轴 9 / 刻度 8 / 注释 7.5 / panel 10）
- **图内文字纪律**: 图内只保留**图例**与 `data-sparse`、`No stable breakpoint detected` 两条短说明；
  数值、样本量、时间范围、断点证据、稀疏区阈值与观测数**一律进图注**（图注按 §1.6.7 八段顺序）
- **区间口径裁决（2026-10-02）**: 主图与图注一律 **section-clustered CR1**；日期聚类仅作敏感性留档
  （`fig04_annotations.csv` 的 `sensitivity_*` 键），**不得**出现在图面或图注


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path


def _resolve_scripts_dir() -> Path:
    """定位 figure/scripts，使本册在**任何内核 cwd** 下都可运行。

    兼容场景：cwd = figure/notebooks（nbconvert 默认）、figure/、仓库根（VS Code / 任意
    启动位置的默认行为）、以及更上层目录；最后回退到本仓库的绝对路径，保证一定能找到
    fig_common.py（旧版写死 `Path.cwd().parent / "scripts"`，只在 notebooks/ 下可用）。
    """
    here = Path.cwd()
    cands = [here / "scripts", here.parent / "scripts"]
    cands += [p / "figure" / "scripts" for p in (here, *here.parents)]
    cands.append(Path(r"D:\NJU_folder\final_update\15_paper\figure\scripts"))
    for c in cands:
        if (c / "fig_common.py").exists():
            return c.resolve()
    raise RuntimeError("未找到 figure/scripts/fig_common.py：请从本仓库、figure/ 或 "
                       "figure/notebooks/ 任一目录启动内核，或检查仓库路径")


SCRIPTS_DIR = _resolve_scripts_dir()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))
print("scripts dir:", SCRIPTS_DIR)

from fig_common import *          # 样式、色板、导出、核验（唯一来源，勿在本册硬编码样式）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f">>> Figure 4 notebook | design system {DESIGN_VERSION} | notebook API v{NOTEBOOK_API} "
      f"| 2×2 面板 + NJU purple")
print(">>> 若本行未出现、或版本号低于上面的值，说明内核里跑的是**旧版 notebook**："
      "请在编辑器执行 Revert File（VS Code）/ 关闭标签后重新打开本文件")

# ---- 本图配置 -------------------------------------------------------------
FIG_ID = "Figure 4"
IS_DRAFT = os.environ.get("FIG_DRAFT", "1") == "1"   # 1 -> out/draft；0 -> out/submission
DATA_FILES = ["fig04_curve_grids.csv", "fig04_threshold_points.csv", "fig04_exposure_hist.csv",
              "fig04_annotations.csv", "fig04_plot_windows.csv"]
t0 = time.time()
meta = figure_meta(FIG_ID)
print(f"FIG_ID={FIG_ID}  stem={meta['filename_stem']}  IS_DRAFT={IS_DRAFT}")
print("data dir   :", FIG_DATA, "| 数据文件齐备:", all((FIG_DATA / f).exists() for f in DATA_FILES))
print("输出目录:", FIG_OUT_DRAFT if IS_DRAFT else FIG_OUT_SUB)


In [ ]:
# ---- 数据加载与核验（不匹配即中断）---------------------------------------
report = verify_figure_data(FIG_ID, data_files=DATA_FILES)
grids = pd.read_csv(FIG_DATA / "fig04_curve_grids.csv")
thr = pd.read_csv(FIG_DATA / "fig04_threshold_points.csv")
hist = pd.read_csv(FIG_DATA / "fig04_exposure_hist.csv")
ann = pd.read_csv(FIG_DATA / "fig04_annotations.csv").set_index("key")["value"]
windows = pd.read_csv(FIG_DATA / "fig04_plot_windows.csv")

# 结构性自检（防止冻结文件被替换成"形似而数不同"的内容）
assert set(grids.panel) == {"a", "b"} and len(grids) == 403
# 版本守卫：冻结文件必须与当前绘图代码（core.build_figure 读取的键）匹配；
# 若这里报错，先按提示重跑 extract；若报错提到 STYLE/PALETTE 等旧代码标识，则是编辑器仍持有旧版 notebook。
_needed = {"x_model", "rr", "ci_lo", "ci_hi", "panel"}
assert _needed.issubset(grids.columns), (
    f"fig04_curve_grids.csv 缺少列 {sorted(_needed - set(grids.columns))}；"
    "请先运行 python figure\scripts\extract_fig04_data.py 重新冻结")
for _k in ("sparse_precip_threshold_mm", "sparse_visibility_threshold_m", "reference_visibility_m",
           "n_estimation", "spline_df_a"):
    assert _k in ann.index, (
        f"fig04_annotations.csv 缺少键 '{_k}'；数据与当前代码不匹配（可能是旧版冻结文件）。"
        "请先运行 python figure\scripts\extract_fig04_data.py 重新冻结，"
        "或从磁盘重新加载 figure\notebooks\fig04_exposure_response.ipynb（若你打开的是旧版 notebook）")
assert (grids.groupby("panel").size() >= 201).all()
ref = grids[grids.is_reference]
assert len(ref) == 2 and ref.panel.nunique() == 2, "每个 panel 只应有一个参考格点"
assert np.allclose(ref.rr, 1.0, atol=1e-6), "参考点 RR 必须为 1"
assert set(windows.panel) == {"a", "b"}
n_est = int(float(ann["n_estimation"]))
assert len(hist) == 96 and (hist.groupby("panel")["count"].sum() == n_est).all()
assert float(ann["assert_failures"]) == 0
print("核验通过:", report["checked"])
print()
print(pd.DataFrame({
    "值": ["估计样本 n", "参考（降水）", "参考（能见度，晴天中位数）", "稀疏阈值（降水 > P95）",
           "稀疏阈值（能见度 < P5）", "稀疏区观测数（a）", "稀疏区观测数（b）",
           "样条联合 Wald p（a）", "样条联合 Wald p（b）"],
    "取值": [ann["n_estimation"], f"{float(ann['reference_precip_mm']):g} mm",
             f"{float(ann['reference_visibility_m']):,.1f} m",
             f"{float(ann['sparse_precip_threshold_mm']):.3f} mm",
             f"{float(ann['sparse_visibility_threshold_m']):.1f} m",
             ann["n_obs_sparse_precip_estimation_sample"],
             ann["n_obs_sparse_visibility_estimation_sample"],
             f"{float(ann['joint_wald_p_a']):.3g}", f"{float(ann['joint_wald_p_b']):.3g}"]}).to_string(index=False))


In [ ]:
# ---- 绘图逻辑：两块曲线面板 + 各自底部的暴露分布条带 -----------------------
# 面板装配代码放在同图模块 fig04_exposure_response_core.build_figure() 内，
# 使命令行试绘（--draft/--submission）与本册出图共用唯一实现，避免两处拷贝漂移。
from fig04_exposure_response_core import build_figure, xplot

fig = build_figure()
for ax in fig.get_axes():
    print(f"{ax.get_label():>6s}  xlim={tuple(round(float(v), 4) for v in ax.get_xlim())}  "
          f"ylim={tuple(round(float(v), 4) for v in ax.get_ylim())}")
fig


In [ ]:
# ---- 导出双版本 + caption + qc -------------------------------------------
written = export_figure(fig, FIG_ID, is_draft=IS_DRAFT)
print("\n".join(str(p) for p in written))

from fig04_exposure_response_core import CAPTION

cap = write_caption(FIG_ID, CAPTION,
                    title="Figure 4. Exposure–response relationships for continuous weather exposures.")
print(cap)

qc = fig_qc_dump(FIG_ID, {
    "status": "ok", "is_draft": IS_DRAFT, "seconds": round(time.time() - t0, 1),
    "data_checked": report["checked"], "data_files": DATA_FILES,
    "outputs": [{"path": str(p), "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in written],
    "caption": str(cap),
})
print(qc)

# 投稿版凭据：只在非草稿运行时写入，避免被后续草稿重跑覆盖（发布门禁以它为准）
if not IS_DRAFT:
    rec = fig_submission_record(FIG_ID, written, cap,
                                {"data_checked": report["checked"], "data_files": DATA_FILES})
    print("submission record:", rec)
